# Evaluasi Invoice Fraud & Three-Way Match Agent

Notebook ini memuat dataset contoh, menjalankan seluruh pemeriksaan, dan menilai hasilnya terhadap label kebenaran di `samples/labels.json`. Output pada setiap sel adalah output nyata dari run yang sama dengan yang dipakai untuk README.

In [ ]:
import sys
sys.path.insert(0, '..')

import checks, loader, run_eval, scoring

dataset = loader.load_dataset('../samples')
labels = loader.load_labels('../samples')
print(f"Vendor: {len(dataset.vendors)} | PO: {len(dataset.pos)} | "
      f"BAST: {len(dataset.receipts)} | Invoice: {len(dataset.invoices)}")

Vendor: 10 | PO: 24 | BAST: 24 | Invoice: 148


In [ ]:
analysis = checks.run_checks(dataset)
scores = scoring.score_invoices(dataset, analysis)
# cetak distribusi status dan temuan per jenis

Total temuan: 50
BERSIH: 118 invoice
PERINGATAN: 17 invoice
TAHAN BAYAR: 13 invoice
angka_bulat: 9 invoice tertandai
benford: 9 invoice tertandai
ganda_eksak: 6 invoice tertandai
ganda_samar: 6 invoice tertandai
kelebihan_volume: 4 invoice tertandai
pecah_invoice: 7 invoice tertandai
rekening_sama: 0 invoice tertandai
selisih_harga: 4 invoice tertandai
tanpa_po: 3 invoice tertandai


## Tabel evaluasi terhadap label kebenaran

Invoice dinilai terdeteksi untuk satu jenis fraud bila agent menghasilkan temuan berjenis sama dengan labelnya.

In [ ]:
predicted = run_eval.predicted_by_invoice(analysis)
results = run_eval.compute_metrics(labels, predicted)
# cetak tabel precision, recall, F1 per jenis dan keseluruhan

Jenis                  TP   FP   FN  Precision   Recall       F1
ganda_eksak             6    0    0      1.000    1.000    1.000
ganda_samar             6    0    0      1.000    1.000    1.000
pecah_invoice           7    0    0      1.000    1.000    1.000
tanpa_po                3    0    0      1.000    1.000    1.000
kelebihan_volume        4    0    0      1.000    1.000    1.000
selisih_harga           4    0    0      1.000    1.000    1.000
angka_bulat             9    0    0      1.000    1.000    1.000
melebihi_nilai_po       0    0    0          -        -        -
KESELURUHAN            39    0    0      1.000    1.000    1.000


## Uji Benford

Distribusi digit pertama seluruh nominal invoice dibandingkan dengan distribusi Benford memakai chi-square dan MAD Nigrini.

In [ ]:
stats = checks.benford_stats([i.total for i in dataset.invoices])
# cetak proporsi aktual vs harapan, chi-square, dan MAD

n = 148
Digit | Aktual | Harapan Benford
  1   | 0.270  | 0.301
  2   | 0.196  | 0.176
  3   | 0.128  | 0.125
  4   | 0.101  | 0.097
  5   | 0.068  | 0.079
  6   | 0.014  | 0.067
  7   | 0.095  | 0.058
  8   | 0.047  | 0.051
  9   | 0.081  | 0.046
Chi-square = 14.90
MAD = 0.0221 (tidak sesuai (nonconformity))
Grafik perbandingan aktual vs harapan ada di ../docs/img/benford.png
